# Functional Evaluation: LoRA Adapter on MI300X Hardware

**Notebook:** `FunctionalLoraTest.ipynb`  
**Base Model:** `Qwen2.5-Coder-7B-Instruct`  
**Adapter:** `Daleth-hb/qwen-cuda2hip-lora`  
**Hardware:** AMD Instinct MI300X (ROCm 7.2.3)

### Objective
Verify the functional capabilities of the LoRA adapter in a high-performance computing (HPC) environment. This test focuses on specialized kernel generation, memory optimization via vectorization, and general debugging logic.

In [1]:
!pip install huggingface_hub
!pip install --upgrade pip
!pip install transformers==5.8.0 datasets accelerate trl peft sentencepiece bitsandbytes

## 1. System & Environment Audit
Before testing, we verify the ROCm stack and hardware visibility. The presence of the **MI300X** allows us to test specialized optimizations that are not available on standard consumer hardware.

In [2]:
import torch

print(torch.__version__)
print(torch.version.hip)
print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

2.10.0+rocm7.2.3.git1a270074
7.2.53211
True
AMD Instinct MI300X VF


In [ ]:
from huggingface_hub import login

login("hf_your_token")


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-Coder-7B-Instruct" 
lora_id = "Daleth-hb/qwen-cuda2hip-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, lora_id)
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████████████████████████████| 339/339 [00:01<00:00, 192.45it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

## 2. Test Harness: HIP Code Generation
We implement a robust generation function with a strict system prompt. This ensures the model acts as an **Elite ROCm/HIP Developer**, forcing it to output raw C++ code without unnecessary conversational filler, optimized for the MI300X.

In [5]:


def generate_hip_code(user_prompt, temperature=0.1, max_tokens=512):
    system_prompt = """You are an elite AMD ROCm/HIP GPU developer. Write compiled C++ code for MI300X.
STRICT RULES:
1. ONLY HIP API (<hip/hip_runtime.h>).
2. NO CPU CODE. NO OPENCL.
3. Output ONLY the raw C++ code. Do not explain. Stop writing after the code ends."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            repetition_penalty=1.1, 
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    response_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]
    
    return response.strip()

## 3. Execution of Test Cases
We are evaluating three distinct scenarios:
1. **Syntax Mapping:** Direct translation of CUDA `__global__` kernels to HIP.
2. **Advanced Optimization:** Implementation of `float4` vectorization to maximize memory throughput on the MI300X bus.
3. **General Debugging:** Ability to identify and fix common memory errors (Segmentations Faults) in a C++ context.

In [6]:

test_1 = """Convert this CUDA kernel to HIP (ROCm):
__global__ void add(int *a, int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}"""

test_2 = """Optimize this kernel for AMD MI300X GPU:
for (int i = 0; i < N; i++) {
    C[i] = A[i] * B[i];
}
Write a HIP kernel. Use HIP vector types like float4 to optimize memory loads/stores on the GPU. DO NOT use CPU AVX instructions."""

test_3 = """This code crashes with segmentation fault:
int *p;
*p = 10;
Explain why and fix it.""" 


In [7]:

print("="*50)
print("TEST 1: CONVERTION CUDA -> HIP (7B + LoRA)")
print("="*50)
print(generate_hip_code(test_1))
print("\n")

print("="*50)
print("TEST 2: VECTORIZACIÓN MI300X (7B + LoRA)")
print("="*50)
print(generate_hip_code(test_2))
print("\n")

print("="*50)
print("TEST 3: SEGFAULT C SOLVE (7B + LoRA)")
print("="*50)
print(generate_hip_code(test_3))

TEST 1: CONVERTION CUDA -> HIP (7B + LoRA)
```cpp
#include <hip/hip_runtime.h>

__global__ void add(int *a, int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}
```


TEST 2: VECTORIZACIÓN MI300X (7B + LoRA)
```cpp
#include <hip/hip_runtime.h>

__global__ void multiplyKernel(float* A, float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        float4 a_vec = *(float4*)(A + idx);
        float4 b_vec = *(float4*)(B + idx);
        float4 c_vec = make_float4(a_vec.x * b_vec.x, a_vec.y * b_vec.y, a_vec.z * b_vec.z, a_vec.w * b_vec.w);
        *(float4*)(C + idx) = c_vec;
    }
}

void launchMultiplyKernel(hipStream_t stream, float* d_A, float* d_B, float* d_C, int N) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;
    multiplyKernel<<<blocksPerGrid, threadsPerBlock, 0, stream>>>(d_A, d_B, d_C, N);
}
```


TEST 3: SEGFAULT C SOLVE (7B + LoRA)
The code crashes because `p` i

## 4. Results and Analysis
* **Vectorization Efficiency:** The model successfully utilized `float4` and `make_float4`, showing awareness of the AMD hardware's capability for vectorized loads/stores.
* **Instruction Adherence:** The adapter respected the "NO CPU CODE" constraint, delivering pure GPU kernels.
* **MI300X Readiness:** The generated code properly includes `<hip/hip_runtime.h>`, making it immediately compatible with the `hipcc` compiler.